# FER2013 — არქიტექტურა 2: MediumCNN

4 conv layer + BatchNorm. მიზანი: overfitting-ის დემონსტრაცია და შემდეგ მისი გამოსწორება.

| Run | ცვლილება | მოსალოდნელი შედეგი |
|---|---|---|
| arch2_no_dropout | Dropout=0 | Overfit: train >> val |
| arch2_dropout_0.5 | Dropout=0.5 | gap მცირდება |
| arch2_dropout_augment | +Augmentation | კიდევ უკეთესი |
| arch2_full | +Class weights + Cosine LR | საუკეთესო |


## დაყენება და იმპორტი

In [ ]:
!pip install -q wandb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

## WandB კავშირი

In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))

## მონაცემების ჩატვირთვა

In [ ]:
CSV_PATH = '/kaggle/input/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.csv'
EMOTION_LABELS = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy', 4:'Sad', 5:'Surprise', 6:'Neutral'}

class FER2013Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.pixels    = dataframe['pixels'].tolist()
        self.labels    = dataframe['emotion'].tolist()
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = Image.fromarray(
            np.array(self.pixels[idx].split(), dtype=np.uint8).reshape(48, 48), mode='L'
        )
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)


def get_transform(augment=False):
    norm = transforms.Normalize([0.5], [0.5])
    if augment:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomCrop(48, padding=4),
            transforms.ToTensor(), norm
        ])
    return transforms.Compose([transforms.ToTensor(), norm])


df       = pd.read_csv(CSV_PATH)
train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
val_df   = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)

counts        = train_df['emotion'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 7

def make_loaders(batch_size=64, augment=False):
    tr = DataLoader(FER2013Dataset(train_df, get_transform(augment)), batch_size, shuffle=True,  num_workers=2)
    vl = DataLoader(FER2013Dataset(val_df,   get_transform(False)),   batch_size, shuffle=False, num_workers=2)
    return tr, vl

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

## არქიტექტურა 2: MediumCNN

```
Input (B, 1, 48, 48)
  Conv(1→32)   + BN + ReLU + MaxPool   (B, 32,  24, 24)
  Conv(32→64)  + BN + ReLU + MaxPool   (B, 64,  12, 12)
  Conv(64→128) + BN + ReLU + MaxPool   (B, 128,  6,  6)
  Conv(128→256)+ BN + ReLU + MaxPool   (B, 256,  3,  3)
  Flatten → FC(2304→512) → Dropout → FC(512→128) → Dropout → FC(128→7)
```

In [ ]:
class MediumCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,   32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 128),         nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


m = MediumCNN()
print(f'პარამეტრები: {sum(p.numel() for p in m.parameters()):,}')

## Forward და Backward Pass შემოწმება

In [ ]:
m = MediumCNN().to(DEVICE)
m.eval()
dummy = torch.randn(4, 1, 48, 48).to(DEVICE)
with torch.no_grad():
    out = m(dummy)
assert out.shape == (4, 7)
assert not torch.isnan(out).any()
print(f'Forward pass OK — output shape: {out.shape}')

In [ ]:
m.train()
m.zero_grad()
dummy_labels = torch.randint(0, 7, (4,)).to(DEVICE)
nn.CrossEntropyLoss()(m(dummy), dummy_labels).backward()

print(f'{'Layer':<35} {'Mean |grad|':>12}')
print('-' * 50)
names, avg_g = [], []
for name, p in m.named_parameters():
    if p.requires_grad and p.grad is not None:
        g = p.grad.abs().mean().item()
        print(f'{name:<35} {g:>12.2e}')
        names.append(name); avg_g.append(g)

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(avg_g)), avg_g, color='steelblue')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=60, ha='right', fontsize=7)
ax.set_title('Gradient Flow — MediumCNN')
plt.tight_layout()
plt.show()

m.zero_grad()
print('Backward pass OK')

## სწავლების ციკლი

In [ ]:
def compute_grad_norm(model):
    return sum(p.grad.detach().norm(2).item()**2
               for p in model.parameters() if p.grad is not None) ** 0.5

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    tl, correct, total, gn = 0.0, 0, 0, 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        gn = compute_grad_norm(model)
        optimizer.step()
        tl      += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return tl / total, correct / total, gn

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tl, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        tl      += loss.item() * imgs.size(0)
        preds    = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return tl / total, correct / total, all_preds, all_labels


def run_experiment(run_name, config):
    run = wandb.init(
        project='fer2013', name=run_name, config=config,
        tags=['architecture_2', 'medium_cnn'], reinit=True
    )

    t_ldr, v_ldr = make_loaders(batch_size=config['batch_size'], augment=config['augment'])
    model     = MediumCNN(dropout=config['dropout']).to(DEVICE)
    w         = class_weights.to(DEVICE) if config.get('use_class_weights') else None
    criterion = nn.CrossEntropyLoss(weight=w)

    optimizer = (
        torch.optim.Adam(model.parameters(), lr=config['lr'],
                         weight_decay=config.get('weight_decay', 0))
        if config['optimizer'] == 'adam'
        else torch.optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9,
                             weight_decay=config.get('weight_decay', 0))
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])
        if config.get('scheduler') == 'cosine' else None
    )

    best = 0.0
    for epoch in range(1, config['epochs'] + 1):
        tr_l, tr_a, gn         = train_epoch(model, t_ldr, criterion, optimizer)
        vl_l, vl_a, preds, lbs = evaluate(model, v_ldr, criterion)
        if scheduler: scheduler.step()

        wandb.log({
            'epoch': epoch,
            'train/loss': tr_l, 'train/accuracy': tr_a,
            'val/loss':   vl_l, 'val/accuracy':   vl_a,
            'grad_norm':  gn,   'learning_rate':  optimizer.param_groups[0]['lr'],
        }, step=epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:2d} | train={tr_a:.3f} val={vl_a:.3f} gap={tr_a-vl_a:+.3f} | gn={gn:.3f}')

        if vl_a > best:
            best = vl_a
            torch.save(model.state_dict(), f'{run_name}_best.pt')

    cm = confusion_matrix(lbs, preds, labels=list(range(7)))
    wandb.log({'confusion_matrix': wandb.Table(
        columns=['True \\ Pred'] + list(EMOTION_LABELS.values()),
        data=[[EMOTION_LABELS[i]] + r.tolist() for i, r in enumerate(cm)]
    )})
    wandb.summary['best_val_accuracy'] = best
    run.finish()
    print(f'Best val acc: {best:.4f}\n')
    return best, model

## ექსპერიმენტი A — Dropout არ არის (განზრახ overfit)

რეგულარიზაცია მოხსნილია. WandB-ზე: `train/accuracy` ~80-85%, `val/accuracy` ~58% — სხვაობა იზრდება.

In [ ]:
acc_A, model_A = run_experiment('arch2_no_dropout', {
    'architecture': 'medium_cnn', 'dropout': 0.0,
    'augment': False, 'use_class_weights': False,
    'optimizer': 'adam', 'lr': 1e-3, 'weight_decay': 0.0,
    'scheduler': None, 'batch_size': 64, 'epochs': 40
})

## ექსპერიმენტი B — Dropout=0.5

მხოლოდ ერთი ცვლილება: `dropout=0.5`. train/val სხვაობა მცირდება.

In [ ]:
acc_B, model_B = run_experiment('arch2_dropout_0.5', {
    'architecture': 'medium_cnn', 'dropout': 0.5,
    'augment': False, 'use_class_weights': False,
    'optimizer': 'adam', 'lr': 1e-3, 'weight_decay': 0.0,
    'scheduler': None, 'batch_size': 64, 'epochs': 40
})

## ექსპერიმენტი C — Dropout + Augmentation

Dropout-ს ემატება: RandomFlip, RandomRotation, RandomCrop.

In [ ]:
acc_C, model_C = run_experiment('arch2_dropout_augment', {
    'architecture': 'medium_cnn', 'dropout': 0.5,
    'augment': True, 'use_class_weights': False,
    'optimizer': 'adam', 'lr': 1e-3, 'weight_decay': 1e-4,
    'scheduler': None, 'batch_size': 64, 'epochs': 40
})

## ექსპერიმენტი D — სრული რეგულარიზაცია

Class weights (imbalance გამოსწორება) + Cosine LR scheduler.

In [ ]:
acc_D, model_D = run_experiment('arch2_full_regularization', {
    'architecture': 'medium_cnn', 'dropout': 0.5,
    'augment': True, 'use_class_weights': True,
    'optimizer': 'adam', 'lr': 1e-3, 'weight_decay': 1e-4,
    'scheduler': 'cosine', 'batch_size': 64, 'epochs': 40
})

## შედეგები და ანალიზი

In [ ]:
results = {
    'A: No Dropout':          acc_A,
    'B: +Dropout':            acc_B,
    'C: +Augment':            acc_C,
    'D: Full Regularization': acc_D,
}

for k, v in results.items():
    print(f'{k:<28} {v:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(results.keys(), results.values(), color=['#d73027','#fc8d59','#91cf60','#1a9641'])
ax.bar_label(bars, fmt='{:.3f}')
ax.set_title('Architecture 2 — რეგულარიზაციის ეფექტი')
ax.set_ylabel('Best Val Accuracy')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
_, v_ldr = make_loaders(batch_size=64, augment=False)
_, _, preds, labels = evaluate(model_D, v_ldr, nn.CrossEntropyLoss())
print(classification_report(labels, preds, target_names=list(EMOTION_LABELS.values())))